## Imports + augmentaciones seguras para espectrogramas

In [2]:
import os, math, random
from typing import Optional
from PIL import Image

import torch
from torch import nn
from torch.utils.data import DataLoader, random_split

import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping, StochasticWeightAveraging
from pytorch_lightning.loggers import CSVLogger

from torchvision import transforms
from torchvision.transforms import InterpolationMode
from torchvision.datasets import ImageFolder

import timm
from safetensors.torch import load_file as safe_load

from torchmetrics.classification import (
    MulticlassAccuracy, MulticlassPrecision, MulticlassRecall, MulticlassF1Score
)

# reproducibilidad y precisión
pl.seed_everything(42, workers=True)
torch.set_float32_matmul_precision("high")

# --------------------- Augmentaciones amigables a espectrogramas ---------------------
class TimeShift:
    """Desplaza en el eje tiempo (W) una fracción aleatoria del ancho."""
    def __init__(self, max_frac: float = 0.12, p: float = 0.8):
        self.max_frac = max_frac
        self.p = p
    def __call__(self, x: torch.Tensor) -> torch.Tensor:
        if random.random() > self.p:
            return x
        _, _, W = x.shape
        max_shift = max(1, int(W * self.max_frac))
        shift = random.randint(-max_shift, max_shift)
        return torch.roll(x, shifts=shift, dims=2)

class SpecMask:
    """SpecAugment simple: enmascara bandas de tiempo/frecuencia poniéndolas a 0."""
    def __init__(self, time_frac: float = 0.10, freq_frac: float = 0.10,
                 num_time_masks: int = 2, num_freq_masks: int = 2, p: float = 0.8):
        self.time_frac = time_frac
        self.freq_frac = freq_frac
        self.num_time_masks = num_time_masks
        self.num_freq_masks = num_freq_masks
        self.p = p
    def __call__(self, x: torch.Tensor) -> torch.Tensor:
        if random.random() > self.p:
            return x
        C, H, W = x.shape
        # tiempo (W)
        for _ in range(self.num_time_masks):
            w = max(1, int(W * self.time_frac * random.random()))
            s = random.randint(0, max(0, W - w))
            x[:, :, s:s+w] = 0.0
        # frecuencia (H)
        for _ in range(self.num_freq_masks):
            h = max(1, int(H * self.freq_frac * random.random()))
            s = random.randint(0, max(0, H - h))
            x[:, s:s+h, :] = 0.0
        return x

def spectro_transforms(img_size: int = 224):
    """Sin flips ni color jitter; TimeShift + SpecAugment."""
    train_tf = transforms.Compose([
        transforms.Lambda(lambda im: im.convert("RGB")),
        transforms.Resize(int(img_size * 1.05), interpolation=InterpolationMode.BICUBIC),
        transforms.CenterCrop(img_size),
        transforms.ToTensor(),
        TimeShift(max_frac=0.12, p=0.8),
        SpecMask(time_frac=0.10, freq_frac=0.10, num_time_masks=2, num_freq_masks=2, p=0.8),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    eval_tf = transforms.Compose([
        transforms.Lambda(lambda im: im.convert("RGB")),
        transforms.Resize(img_size, interpolation=InterpolationMode.BICUBIC),
        transforms.CenterCrop(img_size),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    return train_tf, eval_tf


Seed set to 42
c:\Users\IALAB\anaconda3\envs\TesisRichiFix\lib\site-packages\torch\__init__.py:1617: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\Context.cpp:85.)
  _C._set_float32_matmul_precision(precision)


## DataModule (split fijo; test solo al final)

In [3]:
class SpectroImageDataModule(pl.LightningDataModule):
    def __init__(
        self,
        train_dir: str,
        test_dir: str,
        img_size: int = 224,
        batch_size: int = 32,
        num_workers: int = 0,   # Windows: 0
        val_split: float = 0.15,
        seed: int = 42
    ):
        super().__init__()
        self.train_dir = train_dir
        self.test_dir = test_dir
        self.img_size = img_size
        self.batch_size = batch_size
        self.num_workers = num_workers
        self.val_split = val_split
        self.seed = seed

        self.train_tf, self.eval_tf = spectro_transforms(img_size)
        self.class_names = None
        self._train = self._val = self._test = None

    def setup(self, stage: Optional[str] = None):
        full_train = ImageFolder(self.train_dir, transform=self.train_tf)
        self.class_names = full_train.classes

        n = len(full_train)
        n_val = int(self.val_split * n)
        n_train = n - n_val
        self._train, self._val = random_split(
            full_train, [n_train, n_val],
            generator=torch.Generator().manual_seed(self.seed)
        )
        # validación con eval_tf
        val_dataset = ImageFolder(self.train_dir, transform=self.eval_tf)
        self._val.dataset = val_dataset
        # test con eval_tf
        self._test = ImageFolder(self.test_dir, transform=self.eval_tf)

    def train_dataloader(self):
        return DataLoader(self._train, batch_size=self.batch_size, shuffle=True,
                          num_workers=self.num_workers, pin_memory=True)

    def val_dataloader(self):
        return DataLoader(self._val, batch_size=self.batch_size, shuffle=False,
                          num_workers=self.num_workers, pin_memory=True)

    def test_dataloader(self):
        return DataLoader(self._test, batch_size=self.batch_size, shuffle=False,
                          num_workers=self.num_workers, pin_memory=True)

## Modelo ViT (12 bloques) + label smoothing + freeze→unfreeze

In [4]:
class ViTTransferLit(pl.LightningModule):
    def __init__(
        self,
        num_classes: int,
        local_pretrained_path: str,          # ruta a .safetensors/.pth
        pretrained_name: str = "vit_base_patch16_224",  # 12 bloques
        img_size: int = 224,
        lr: float = 1e-4,        # <- OJO al LR: 1e-4
        min_lr: float = 1e-6,
        weight_decay: float = 0.05,
        dropout: float = 0.1,
        drop_path: float = 0.0,
        freeze_ratio: float = 1.0,    # congela todo al inicio
        unfreeze_epoch: Optional[int] = 2,  # libera pronto (época 2)
        label_smoothing: float = 0.1,
        warmup_epochs: int = 8         # warmup para 200 épocas
    ):
        super().__init__()
        self.save_hyperparameters()

        # Backbone sin descarga
        self.backbone = timm.create_model(
            pretrained_name, pretrained=False, num_classes=0,
            img_size=img_size, drop_path_rate=drop_path
        )
        in_features = self.backbone.num_features

        # Cargar pesos locales (solo backbone)
        self._load_local_backbone(self.backbone, local_pretrained_path)
        print(f"[ViT] Pesos locales cargados desde: {local_pretrained_path}")

        # Cabeza de clasificación
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(in_features, num_classes))

        # Pérdida con smoothing
        self.criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)

        # Métricas por época
        self.train_acc  = MulticlassAccuracy(num_classes=num_classes)
        self.train_prec = MulticlassPrecision(num_classes=num_classes, average="macro")
        self.train_rec  = MulticlassRecall(num_classes=num_classes, average="macro")
        self.train_f1   = MulticlassF1Score(num_classes=num_classes, average="macro")

        self.val_acc  = MulticlassAccuracy(num_classes=num_classes)
        self.val_prec = MulticlassPrecision(num_classes=num_classes, average="macro")
        self.val_rec  = MulticlassRecall(num_classes=num_classes, average="macro")
        self.val_f1   = MulticlassF1Score(num_classes=num_classes, average="macro")

        # Congelado total al inicio
        self._freeze_all_backbone()
        self.unfreeze_epoch = unfreeze_epoch

    @torch.no_grad()
    def _load_local_backbone(self, model: nn.Module, ckpt_path: str):
        ok = False
        try:
            timm.models.load_checkpoint(model, ckpt_path, strict=False)
            ok = True
        except Exception:
            pass
        if not ok and ckpt_path and ckpt_path.lower().endswith(".safetensors"):
            state = safe_load(ckpt_path, device="cpu")
            state = {k.replace("model.", "").replace("module.", ""): v for k, v in state.items()}
            _ = model.load_state_dict(state, strict=False)
            ok = True
        if not ok and ckpt_path:
            state = torch.load(ckpt_path, map_location="cpu")
            if isinstance(state, dict) and "state_dict" in state:
                state = state["state_dict"]
            state = {k.replace("model.", "").replace("module.", ""): v for k, v in state.items()}
            _ = model.load_state_dict(state, strict=False)

    def _freeze_all_backbone(self):
        for p in self.backbone.parameters():
            p.requires_grad = False

    def _unfreeze_all_backbone(self):
        for p in self.backbone.parameters():
            p.requires_grad = True
        print(f"[ViT] Backbone UNFROZEN epoch {self.current_epoch}")

    # Forward
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.head(self.backbone(x))

    # Hook: unfreeze temprano
    def on_train_epoch_start(self) -> None:
        if self.unfreeze_epoch is not None and self.current_epoch == int(self.unfreeze_epoch):
            self._unfreeze_all_backbone()

    # TRAIN
    def training_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.criterion(logits, y)
        preds = torch.argmax(logits, dim=1)
        self.log("train/loss", loss, on_step=False, on_epoch=True, prog_bar=True)
        self.log("train/accuracy", self.train_acc(preds, y), on_step=False, on_epoch=True, prog_bar=True)
        self.log("train/precision", self.train_prec(preds, y), on_step=False, on_epoch=True)
        self.log("train/recall", self.train_rec(preds, y), on_step=False, on_epoch=True)
        self.log("train/f1", self.train_f1(preds, y), on_step=False, on_epoch=True, prog_bar=True)
        return loss

    # VAL
    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.criterion(logits, y)
        preds = torch.argmax(logits, dim=1)
        self.log("val/loss", loss, on_step=False, on_epoch=True, prog_bar=True)
        self.log("val/accuracy", self.val_acc(preds, y), on_step=False, on_epoch=True, prog_bar=True)
        self.log("val/precision", self.val_prec(preds, y), on_step=False, on_epoch=True)
        self.log("val/recall", self.val_rec(preds, y), on_step=False, on_epoch=True)
        self.log("val/f1", self.val_f1(preds, y), on_step=False, on_epoch=True, prog_bar=True)
        # Duplicamos clave sin slash para monitor/filename
        self.log("val_f1", self.val_f1(preds, y), on_step=False, on_epoch=True)

    # TEST
    def test_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.criterion(logits, y)
        preds = torch.argmax(logits, dim=1)
        acc  = MulticlassAccuracy(num_classes=self.hparams.num_classes)(preds, y)
        prec = MulticlassPrecision(num_classes=self.hparams.num_classes, average="macro")(preds, y)
        rec  = MulticlassRecall(num_classes=self.hparams.num_classes, average="macro")(preds, y)
        f1   = MulticlassF1Score(num_classes=self.hparams.num_classes, average="macro")(preds, y)
        self.log("test/loss", loss, on_step=False, on_epoch=True)
        self.log("test/accuracy", acc, on_step=False, on_epoch=True)
        self.log("test/precision", prec, on_step=False, on_epoch=True)
        self.log("test/recall", rec, on_step=False, on_epoch=True)
        self.log("test/f1", f1, on_step=False, on_epoch=True)

    # Optimizador + LR warmup → cosine
    def configure_optimizers(self):
        decay, no_decay = [], []
        for n, p in self.named_parameters():
            if not p.requires_grad:
                continue
            (no_decay if (p.ndim == 1 or n.endswith(".bias")) else decay).append(p)
        optim = torch.optim.AdamW(
            [{"params": decay, "weight_decay": self.hparams.weight_decay},
             {"params": no_decay, "weight_decay": 0.0}],
            lr=self.hparams.lr, betas=(0.9, 0.999)
        )
        T = self.trainer.max_epochs
        W = int(self.hparams.warmup_epochs)
        cosine_T = max(1, T - W)
        warmup = torch.optim.lr_scheduler.LinearLR(optim, start_factor=0.1, total_iters=max(1, W))
        cosine = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=cosine_T, eta_min=self.hparams.min_lr)
        sched = torch.optim.lr_scheduler.SequentialLR(optim, schedulers=[warmup, cosine], milestones=[W])
        return {"optimizer": optim, "lr_scheduler": {"scheduler": sched, "interval": "epoch"}}

## Callback para imprimir métricas por época (consola)

In [5]:
class EpochMetricsPrinter(pl.Callback):
    def __init__(self):
        super().__init__()
        self._train_keys = ["train/loss", "train/accuracy", "train/precision", "train/recall", "train/f1"]
        self._val_keys   = ["val/loss",   "val/accuracy",   "val/precision",   "val/recall",   "val/f1"]
        self._test_keys  = ["test/loss",  "test/accuracy",  "test/precision",  "test/recall",  "test/f1"]

    @staticmethod
    def _as_float(x):
        try:
            if isinstance(x, torch.Tensor):
                return float(x.detach().cpu().item())
            return float(x)
        except Exception:
            return None

    def _fmt_line(self, epoch, split, metrics, keys):
        parts = []
        for k in keys:
            v = self._as_float(metrics.get(k, None))
            if v is not None:
                parts.append(f"{k.split('/')[-1]}={v:.4f}")
        return f"[Epoch {epoch:03d}] {split.upper()}: " + "  ".join(parts)

    def on_train_epoch_end(self, trainer, pl_module):
        print(self._fmt_line(trainer.current_epoch, "train", trainer.callback_metrics, self._train_keys), flush=True)

    def on_validation_epoch_end(self, trainer, pl_module):
        print(self._fmt_line(trainer.current_epoch, "val", trainer.callback_metrics, self._val_keys), flush=True)

    def on_test_end(self, trainer, pl_module):
        print(self._fmt_line(trainer.current_epoch, "test", trainer.callback_metrics, self._test_keys), flush=True)



## Entrenamiento (ajusta rutas) + SWA + test final

In [9]:
# ==== RUTAS ====
ROOT      = r"C:\Users\IALAB\Documents\GitHub\data\segmentación5seg_LogMel2D\segmentación5seg_LogMel2D"   # <-- CAMBIA
TRAIN_DIR = os.path.join(ROOT, "train")
TEST_DIR  = os.path.join(ROOT, "test")

LOCAL_WEIGHTS = r"C:\Users\IALAB\Documents\GitHub\data\model.safetensors"  # <-- tu archivo descargado (.safetensors o .pth)

# ==== HYPERPARAMS ====
IMG_SIZE    = 224
BATCH_SIZE  = 64
NUM_WORKERS = 0
VAL_SPLIT   = 0.15
MAX_EPOCHS  = 500

# ==== DATAMODULE ====
dm = SpectroImageDataModule(
    train_dir=TRAIN_DIR,
    test_dir=TEST_DIR,
    img_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    val_split=VAL_SPLIT,
    seed=42
)
dm.setup("fit")
print("Clases:", dm.class_names)

# ==== MODELO (OJO LR=1e-4) ====
model = ViTTransferLit(
    num_classes=len(dm.class_names),
    local_pretrained_path=LOCAL_WEIGHTS,
    pretrained_name="vit_base_patch16_224",
    img_size=IMG_SIZE,
    lr=1e-4,            # <- LR definido aquí
    min_lr=1e-6,
    weight_decay=0.05,
    dropout=0.1,
    drop_path=0.0,
    freeze_ratio=1.0,
    unfreeze_epoch=2,
    label_smoothing=0.1,
    warmup_epochs=8
)
print("Bloques Transformer:", len(model.backbone.blocks))

# ==== CALLBACKS & LOGGER ====
monitor = "val_f1"  # usamos la clave duplicada sin slash
ckpt_cb = ModelCheckpoint(
    monitor=monitor, mode="max",
    save_top_k=3, save_last=True,
    filename="vit12-{epoch:03d}-{val_f1:.4f}"
)
es_cb   = EarlyStopping(monitor=monitor, mode="max", patience=30, verbose=True)
swa_cb  = StochasticWeightAveraging(swa_lrs=5e-6, swa_epoch_start=int(MAX_EPOCHS*0.9))
logger  = CSVLogger(save_dir="logs", name="vit12_transfer")

precision = "16-mixed" if torch.cuda.is_available() else 32

trainer = pl.Trainer(
    max_epochs=MAX_EPOCHS,
    accelerator="auto",
    devices=1,
    precision=precision,
    gradient_clip_val=1.0,
    log_every_n_steps=10,
    enable_progress_bar=True,       # facilita ver los prints
    accumulate_grad_batches=2,       # batch efectivo mayor, más estable
    callbacks=[ckpt_cb, es_cb, swa_cb, EpochMetricsPrinter()],
    logger=logger
)

# ==== ENTRENAR ====
trainer.fit(model, datamodule=dm)

# ==== TEST FINAL con el mejor checkpoint disponible ====
best_path = ckpt_cb.best_model_path or ckpt_cb.last_model_path
print("Best checkpoint:", best_path)
trainer.test(ckpt_path=best_path, datamodule=dm)


Clases: ['Chimayche', 'Cumbia andina', 'Diablada', 'Huaylash', 'Huayno', 'Muliza', 'Santiago', 'Tinku', 'Tunantada', 'Yaravi (Harawi)']


Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


[ViT] Pesos locales cargados desde: C:\Users\IALAB\Documents\GitHub\data\model.safetensors
Bloques Transformer: 12


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

   | Name       | Type                | Params | Mode 
------------------------------------------------------------
0  | backbone   | VisionTransformer   | 85.8 M | train
1  | head       | Sequential          | 7.7 K  | train
2  | criterion  | CrossEntropyLoss    | 0      | train
3  | train_acc  | MulticlassAccuracy  | 0      | train
4  | train_prec | MulticlassPrecision | 0      | train
5  | train_rec  | MulticlassRecall    | 0      | train
6  | train_f1   | MulticlassF1Score   | 0      | train
7  | val_acc    | MulticlassAccuracy  | 0      | train
8  | val_prec   | MulticlassPrecision | 0      | train
9  | val_rec    | MulticlassRecall    | 0      | train
10 | val_f1     | MulticlassF1Score   | 0      | train
------------------------------------------------------------
7.7 K     Trainable params
85.8 M    Non-trainable params
85.8 M    Total params
343.225   Total estimated model params size (MB)
288       Modules in train mode
0         Mod

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

...

Monitored metric val_f1 did not improve in the last 30 records. Best score: 0.546. Signaling Trainer to stop.


Best checkpoint: logs\vit12_transfer\version_1\checkpoints\vit12-epoch=085-val_f1=0.5458.ckpt


Restoring states from the checkpoint path at logs\vit12_transfer\version_1\checkpoints\vit12-epoch=085-val_f1=0.5458.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loaded model weights from the checkpoint at logs\vit12_transfer\version_1\checkpoints\vit12-epoch=085-val_f1=0.5458.ckpt
c:\Users\IALAB\anaconda3\envs\TesisRichiFix\lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:433: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=19` in the `DataLoader` to improve performance.


Testing: |          | 0/? [00:00<?, ?it/s]

RuntimeError: Encountered different devices in metric calculation (see stacktrace for details). This could be due to the metric class not being on the same device as input. Instead of `metric=MulticlassAccuracy(...)` try to do `metric=MulticlassAccuracy(...).to(device)` where device corresponds to the device of the input.

In [1]:
import platform, sys
print(platform.platform)
print("Python:", sys.version)
import torch
print("Torch: ", torch.__version__)
print("CUDA: ", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU: ", torch.cuda.get_device_name(0))

<function platform at 0x000002217D2ECF70>
Python: 3.10.19 | packaged by conda-forge | (main, Oct 22 2025, 22:23:22) [MSC v.1944 64 bit (AMD64)]
Torch:  2.9.0+cu128
CUDA:  True
GPU:  NVIDIA GeForce RTX 3080
